In [9]:
import os
from dotenv import load_dotenv
load_dotenv()

# 2. Now it is safe to import Playwright
from playwright.async_api import async_playwright
import subprocess
import sys 
import time
import requests 

chrome_exec = os.environ.get('CHROME_EXEC')
chrome_rdp = os.environ.get('REMOTE_DEBUGGING_PORT')
MAX_ITERS = os.environ.get('MAX_LOAD_SEC', 10)

if (not chrome_exec) or (not chrome_rdp):
    raise RuntimeError('Define config in .env')

browser_init = [chrome_exec, '--force-device-scale-factor=1', f'--remote-debugging-port={chrome_rdp}', '--user-data-dir=chrome-profile']

def await_rdp_load():
    for _ in range(int(MAX_ITERS)):
        try:
            response = requests.get(f"http://localhost:{chrome_rdp}/json/version", timeout=1)
            if response.status_code == 200:
                print('Browser connected.')
                break
        except requests.RequestException:
            print('Connecting to browser...')
            pass
        time.sleep(1)


In [ ]:
process = subprocess.Popen(
    browser_init,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    start_new_session=True, # only for workbook
)

await_rdp_load()

# browser control 
async with async_playwright() as p:
    browser = await p.chromium.connect_over_cdp(f"http://localhost:{chrome_rdp}")
    browser_session = browser.contexts[0]
    new_tab = browser_session.pages[0]
    await new_tab.goto(url=r'https://chatgpt.com/')
    print('Connected to ' + new_tab.url)
    editor = new_tab.locator('[contenteditable="true"]')
    await editor.fill("Hi, tell me about LLMs")
    

process.kill()

Connecting to browser...
Browser connected.
Connected to https://chatgpt.com/


TimeoutError: Locator.fill: Timeout 30000ms exceeded.
Call log:
  - waiting for locator("[contenteditable=\"true\"]")


In [18]:
process.kill()